# TPPP evidence per counterparty, `pkg_tppp_cpty_evidence.ipynb`

Builds `pkg_tppp_cpty_evidence_monthly`: one row per counterparty entity (`ent_id`, the ent_l2 of
`pkg_cpty_metrics_monthly`) per month, with evidence that monthly averages cannot show. The TPPP Prospect Finder app
reads it when it exists and adds the measures to its model.

| measure in the app | built from | what it shows |
|---|---|---|
| money out right after money in | daily edges: money out to PNC clients matched by at least as much money in from them in the `PASS_DAYS` calendar days before (same day included) | pass-through, as when an employer funds a payroll that goes out a day or two later |
| money out on its 4 biggest days | daily edges: share of the month's money out sent on its `TOP_DAYS` biggest days | payroll pays on a few paydays; a merchant processor settles every day |
| money out on its main weekday | daily edges: share of money out on its most common weekday | weekly or biweekly pay runs |
| paying organisations that also pay a registered TPPP | client pairs: organisations paying it this month that also pay a registered TPPP at PNC this month | it serves the same kind of client as the TPPPs |
| new paying organisations that just left a registered TPPP | client pairs: organisations paying it this month and not in the `SWITCH_LOOKBACK` months before, that paid a registered TPPP in those months and not this month | it took over the job the TPPP did for them |

The table stores sums and counts, not ratios, so the app can add any months together (ratio of sums).

**Scope.** Counterparties with eligible metrics, a business-like name and at least `MIN_CLIENTS` PNC clients in the
month. Self-transfers are left out. Client-overlap measures count organisations (party type O) only.
**Resumable.** A month is skipped when its partition exists with the current `VERSION`, unless `FORCE = True`.
**Run time.** Each month reads one month of daily edges and four months of client pairs.

## §0 Setup

In [ ]:
# §0a  Session and compat shims
import os, re, json, time, math, warnings, datetime as dt, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
with warnings.catch_warnings():                 # hasattr on removed aliases warns in NumPy 2
    warnings.simplefilter("ignore")
    for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                            ("uint0", np.uintp), ("bool", np.bool_), ("object", object)]:
        if not hasattr(np, _alias):
            setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
from IPython.display import display, HTML
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.utils import AnalysisException

spark = (SparkSession.builder
         .appName("pkg_tppp_cpty_evidence")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.executor.memoryOverhead", "2g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.ansi.enabled", "false")
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config and helpers
DB             = "bdahd01p_dlcdi1_cdi_tm"
CUSTOMER_TABLE = "dsihd01p_dsi.vector_customer"
TPPP_TYPE      = "Third-Party Payment/Payroll Processor"
T_CPTY_MET     = f"{DB}.pkg_cpty_metrics_monthly"
T_MAP          = f"{DB}.pkg_cpty_entity_map_v2"
T_EDGE_D       = f"{DB}.pkg_edge_daily"
T_PAIR         = f"{DB}.pkg_cust_pair_month_v3"
T_CUST_DIM     = f"{DB}.pkg_cust_dim"
OUT_NAME       = "pkg_tppp_cpty_evidence_monthly"
T_OUT          = f"{DB}.{OUT_NAME}"

RUN_START, RUN_END = "2025-05", "2026-08"   # covers 2026 periods and the previous period the app compares with
MIN_CLIENTS     = 3            # PNC clients in the month, either way (the app scores 5+ a month on average)
CLASSES         = ["business", "unknown"]
PASS_DAYS       = 4            # calendar days before a payment out (same day included): about 2 business days
TOP_DAYS        = 4            # biggest days of the month for payday concentration
SWITCH_LOOKBACK = 3            # months before: "new" payer, and "left a registered TPPP"
MIN_MATCH_RATE  = 0.5          # fail if fewer registered TPPP power ids than this reach pkg_cust_dim
FORCE           = False        # True: rebuild months that already exist with this VERSION
VERSION         = "tppp_evidence_v1"

def add_months(m, k):
    y, mo = int(m[:4]), int(m[5:]) + k
    y, mo = y + (mo - 1) // 12, (mo - 1) % 12 + 1
    return f"{y:04d}-{mo:02d}"

def month_range(a, b):
    out, m = [], a
    while m <= b:
        out.append(m)
        m = add_months(m, 1)
    return out

RUN_MONTHS = month_range(RUN_START, RUN_END)

def table_exists(t):
    try:
        spark.table(t).columns
        return True
    except AnalysisException:
        return False

def partitions(t):
    out = set()
    if not table_exists(t):
        return out
    for r in spark.sql(f"SHOW PARTITIONS {t}").collect():
        for kv in r[0].split("/"):
            if kv.startswith("month="):
                out.add(kv.split("=", 1)[1])
    return out

def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")

def norm_pwr(c):
    # power id as a 10-digit string with leading zeros; '123.0' -> '0000000123' (same rule as the TPPP notebooks)
    s = F.regexp_replace(F.trim(c.cast("string")), r"\.0+$", "")
    return F.when(F.length(s) > 0, F.lpad(s, 10, "0"))

LOC = db_location(DB)
OUT_LOC = f"{LOC}/{OUT_NAME}"

# ---- display: one output per cell ----
def _fmt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else ""
    if isinstance(v, str):
        return v
    if not isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    v = float(v)
    if v.is_integer():
        return f"{v:,.0f}"
    return f"{v:,.2f}" if abs(v) >= 1 else f"{v:.3f}"

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 8px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt td:nth-child(1){text-align:left}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#6f6e69;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        elif b[1] is not None:
            df = b[1].copy()
            for c in df.columns:
                df[c] = [_fmt(v) for v in df[c].tolist()]
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(df.to_html(index=False, escape=True, border=0, classes="pkgt"))
    # tex2jax_ignore / mathjax_ignore: JupyterLab must not read text between two dollar signs as math
    return HTML(_CSS + "<div class='tex2jax_ignore mathjax_ignore' style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>"
                + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

show("§0b Config", ("Run", pd.DataFrame([{"months": f"{RUN_MONTHS[0]} to {RUN_MONTHS[-1]} ({len(RUN_MONTHS)})",
                                          "output": T_OUT, "location": OUT_LOC, "version": VERSION, "force": FORCE}])))

## §1 Inputs

Checks every input table and column, then finds the registered TPPPs: latest `load_date` snapshot of
`vector_customer`, power ids normalised to 10 digits on both sides of the join to `pkg_cust_dim`.

In [ ]:
# §1  Inputs and registered TPPPs
need = {T_CPTY_MET: {"ent_id", "month", "eligible", "name_class", "n_cust_all", "amt_in", "amt_out", "ent_name"},
        T_MAP: {"cpty_id", "ent_l2"},
        T_EDGE_D: {"txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "is_self", "amount", "month"},
        T_PAIR: {"cust", "dir", "scope", "other", "month"},
        T_CUST_DIM: {"mdm_id", "customer_pwr_id", "party_type", "customer_name"},
        CUSTOMER_TABLE: {"cust_pwr_id", "customer_type", "load_date"}}
chk = []
for t, cols in need.items():
    have = set(spark.table(t).columns) if table_exists(t) else set()
    miss = sorted(cols - have)
    chk.append({"table": t, "exists": bool(have), "missing columns": ", ".join(miss)})
chk = pd.DataFrame(chk)
assert chk.exists.all() and (chk["missing columns"] == "").all(), chk.to_string()
miss_m = [m for m in RUN_MONTHS if m not in partitions(T_EDGE_D)]
assert not miss_m, f"{T_EDGE_D}: months missing {miss_m}"

tppp_pwr = (spark.sql(f"select cust_pwr_id from {CUSTOMER_TABLE} where customer_type = '{TPPP_TYPE}' "
                      f"and load_date = (select max(load_date) from {CUSTOMER_TABLE})")
                 .select(norm_pwr(F.col("cust_pwr_id")).alias("pwr")).where(F.col("pwr").isNotNull()).distinct())
R = (spark.table(T_CUST_DIM).select("mdm_id", "customer_name", norm_pwr(F.col("customer_pwr_id")).alias("pwr"))
          .join(tppp_pwr, "pwr").select("mdm_id", "customer_name").distinct())
Rp = R.toPandas()
n_pwr = tppp_pwr.count()
assert len(Rp) > 0, "no registered TPPP reached pkg_cust_dim"
assert Rp.mdm_id.nunique() >= MIN_MATCH_RATE * n_pwr, f"only {Rp.mdm_id.nunique()} of {n_pwr} TPPP power ids matched"
R_IDS = spark.createDataFrame(Rp[["mdm_id"]].drop_duplicates()).withColumnRenamed("mdm_id", "tppp_id")
ORGS = spark.table(T_CUST_DIM).where(F.col("party_type") == "O").select(F.col("mdm_id").alias("cust")).distinct()
show("§1 Inputs", ("Tables", chk),
     note(f"Registered TPPPs: {n_pwr} power ids in the latest snapshot, {Rp.mdm_id.nunique()} PNC customers (mdm_id)."),
     note("Examples: " + "; ".join(sorted(Rp.customer_name.dropna().unique())[:8])))

## §2 Counterparties in scope and their account ids

Candidates per month: eligible counterparty metrics, a business-like name and at least `MIN_CLIENTS` PNC clients.
The entity map is reduced to one row per account id (lowest `ent_l2`, the map's duplicate rule) before it is joined,
then kept to the candidates' account ids and written to disk once for all months.

In [ ]:
# §2  Candidates and the account-id map
CAND = (spark.table(T_CPTY_MET)
        .where(F.col("month").isin(RUN_MONTHS) & F.col("eligible") & F.col("name_class").isin(CLASSES)
               & (F.col("n_cust_all") >= MIN_CLIENTS))
        .select("ent_id", "month").distinct()
        .persist(StorageLevel.DISK_ONLY))
cand_n = CAND.groupBy("month").count().toPandas().sort_values("month")
ents = CAND.select("ent_id").distinct()
MAPD = (spark.table(T_MAP).groupBy("cpty_id").agg(F.min("ent_l2").alias("ent_id"))
        .join(ents, "ent_id", "left_semi")
        .persist(StorageLevel.DISK_ONLY))
n_ids = MAPD.count()
n_ents = ents.count()
assert n_ents > 0, "no candidate counterparties: check RUN_MONTHS and the counterparty metrics"
show("§2 Candidates", ("Counterparties per month", cand_n.rename(columns={"count": "counterparties"})),
     note(f"{n_ents:,} distinct counterparties over the run, {n_ids:,} account ids."))

## §3 One month of evidence

`build_month(m)` returns one row per candidate counterparty of month `m`:

| column | definition |
|---|---|
| `in_amt`, `out_amt` | money in from / out to PNC clients in the month (daily edges, self-transfers left out) |
| `pt2_out_amt` | sum over days of the smaller of the day's money out and the money in over that day and the `PASS_DAYS` days before |
| `top4_out_amt` | money out on its `TOP_DAYS` biggest days of the month |
| `wkday_max_out_amt` | money out on its biggest weekday |
| `in_days`, `out_days` | days with money in / out |
| `payer_orgs` | organisations paying it this month |
| `payer_orgs_tppp` | of those, organisations also paying a registered TPPP at PNC this month |
| `new_payer_orgs` | organisations paying it this month and not in the `SWITCH_LOOKBACK` months before |
| `switch_from_tppp` | of those, organisations that paid a registered TPPP in the `SWITCH_LOOKBACK` months before and not this month |

In [ ]:
# §3  build_month
AMT_COLS = ["in_amt", "out_amt", "pt2_out_amt", "top4_out_amt", "wkday_max_out_amt"]
CNT_COLS = ["in_days", "out_days", "payer_orgs", "payer_orgs_tppp", "new_payer_orgs", "switch_from_tppp"]

def build_month(m):
    cand = CAND.where(F.col("month") == m).select("ent_id")
    # ---- daily money in / out per counterparty entity
    e = (spark.table(T_EDGE_D)
         .where((F.col("month") == m) & ~F.coalesce(F.col("is_self"), F.lit(False))
                & (((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CPTY"))
                   | ((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CUST"))))
         .select(F.when(F.col("dst_kind") == "CPTY", F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                 (F.col("dst_kind") == "CPTY").alias("is_in"),
                 F.col("txn_date").cast("date").alias("d"), F.col("amount").cast("double").alias("amt"))
         .join(MAPD, "cpty_id").join(cand, "ent_id"))
    day = (e.groupBy("ent_id", "d")
            .agg(F.sum(F.when(F.col("is_in"), F.col("amt")).otherwise(0.0)).alias("in_d"),
                 F.sum(F.when(~F.col("is_in"), F.col("amt")).otherwise(0.0)).alias("out_d"))
            .withColumn("dn", F.datediff(F.col("d"), F.lit("2000-01-01").cast("date"))))
    w_in = Window.partitionBy("ent_id").orderBy("dn").rangeBetween(-PASS_DAYS, 0)
    w_top = Window.partitionBy("ent_id").orderBy(F.col("out_d").desc(), F.col("dn"))     # deterministic ties
    day = (day.withColumn("in_win", F.sum("in_d").over(w_in))
              .withColumn("matched", F.least(F.col("out_d"), F.col("in_win")))
              .withColumn("r_out", F.row_number().over(w_top)))
    daily = day.groupBy("ent_id").agg(
        F.sum("in_d").alias("in_amt"), F.sum("out_d").alias("out_amt"), F.sum("matched").alias("pt2_out_amt"),
        F.sum(F.when((F.col("r_out") <= TOP_DAYS) & (F.col("out_d") > 0), F.col("out_d")).otherwise(0.0))
         .alias("top4_out_amt"),
        F.sum((F.col("in_d") > 0).cast("int")).alias("in_days"),
        F.sum((F.col("out_d") > 0).cast("int")).alias("out_days"))
    wk = (day.where(F.col("out_d") > 0).groupBy("ent_id", F.dayofweek("d").alias("wd")).agg(F.sum("out_d").alias("o"))
             .groupBy("ent_id").agg(F.max("o").alias("wkday_max_out_amt")))
    # ---- client overlap and switching (organisations paying it / paying a registered TPPP)
    lo = add_months(m, -SWITCH_LOOKBACK)
    pair = spark.table(T_PAIR).where(F.col("month").between(lo, m) & (F.col("dir") == "out"))
    px = (pair.where(F.col("scope") == "ext").select("cust", F.col("other").alias("ent_id"), "month")
              .join(cand, "ent_id").join(ORGS, "cust").distinct())
    pr = (pair.where(F.col("scope") == "int").join(R_IDS, F.col("other") == F.col("tppp_id"))
              .select("cust", "month").distinct())
    px_m = px.where(F.col("month") == m).select("cust", "ent_id")
    px_before = px.where(F.col("month") < m).select("cust", "ent_id").distinct()
    pr_m = pr.where(F.col("month") == m).select("cust").distinct()
    pr_before = pr.where(F.col("month") < m).select("cust").distinct()
    new = px_m.join(px_before, ["cust", "ent_id"], "left_anti")
    switch = new.join(pr_before, "cust", "left_semi").join(pr_m, "cust", "left_anti")
    co = px_m.join(pr_m, "cust", "left_semi")
    cnt = lambda df, name: df.groupBy("ent_id").agg(F.countDistinct("cust").alias(name))
    clients = (cnt(px_m, "payer_orgs").join(cnt(co, "payer_orgs_tppp"), "ent_id", "left")
               .join(cnt(new, "new_payer_orgs"), "ent_id", "left")
               .join(cnt(switch, "switch_from_tppp"), "ent_id", "left"))
    out = (cand.join(daily, "ent_id", "left").join(wk, "ent_id", "left").join(clients, "ent_id", "left")
               .fillna(0.0, subset=AMT_COLS).fillna(0, subset=CNT_COLS))
    return out.select("ent_id", *[F.col(c).cast("double").alias(c) for c in AMT_COLS],
                      *[F.col(c).cast("bigint").alias(c) for c in CNT_COLS], F.lit(VERSION).alias("version"))

def qa(df):
    # invariants that must hold for every row; returns the counts of violations and the month totals
    eps = lambda c: F.col(c) * 1.000001 + 0.01
    r = df.agg(F.count("*").alias("rows"), F.countDistinct("ent_id").alias("ents"),
               F.sum((F.col("pt2_out_amt") > eps("out_amt")).cast("int")).alias("bad_pt2"),
               F.sum((F.col("top4_out_amt") > eps("out_amt")).cast("int")).alias("bad_top4"),
               F.sum((F.col("wkday_max_out_amt") > eps("out_amt")).cast("int")).alias("bad_wkday"),
               F.sum((F.col("payer_orgs_tppp") > F.col("payer_orgs")).cast("int")).alias("bad_co"),
               F.sum(((F.col("switch_from_tppp") > F.col("new_payer_orgs"))
                      | (F.col("new_payer_orgs") > F.col("payer_orgs"))).cast("int")).alias("bad_switch"),
               F.sum("out_amt").alias("out_amt"), F.sum("pt2_out_amt").alias("pt2_out_amt"),
               F.sum("switch_from_tppp").alias("switch_from_tppp")).first().asDict()
    return r

show("§3 build_month ready", note(f"Pass-through window {PASS_DAYS} days, payday concentration on the top {TOP_DAYS} "
                                  f"days, switch lookback {SWITCH_LOOKBACK} months."))

## §4 Build the months

Each month is written to its own partition folder, registered on the table, and checked: no row may break the
invariants (matched, top-day and weekday money never above money out; overlap, new and switching counts nested), and
the month must have rows. Months already built with this `VERSION` are skipped unless `FORCE = True`.

In [ ]:
# §4  Run (resumable by month)
cols_sql = ", ".join([f"{c} DOUBLE" for c in AMT_COLS] + [f"{c} BIGINT" for c in CNT_COLS])
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {T_OUT} (ent_id STRING, {cols_sql}, version STRING) "
          f"PARTITIONED BY (month STRING) STORED AS PARQUET LOCATION '{OUT_LOC}'")
done = partitions(T_OUT)
log, t_all = [], time.time()
handle = display(render(note("starting …")), display_id=True)
for m in RUN_MONTHS:
    if m in done and not FORCE:
        v = spark.table(T_OUT).where(F.col("month") == m).select("version").limit(1).collect()
        if v and v[0][0] == VERSION:
            log.append({"month": m, "status": "kept", "minutes": 0.0})
            continue
    t0 = time.time()
    path = f"{OUT_LOC}/month={m}"
    build_month(m).repartition(8).write.mode("overwrite").parquet(path)
    spark.sql(f"ALTER TABLE {T_OUT} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{path}'")
    spark.catalog.refreshTable(T_OUT)
    r = qa(spark.read.parquet(path))
    bad = {k: v for k, v in r.items() if k.startswith("bad_") and v}
    assert r["rows"] > 0 and r["rows"] == r["ents"], f"{m}: {r['rows']} rows for {r['ents']} counterparties"
    assert not bad, f"{m}: invariant violations {bad}"
    log.append({"month": m, "status": "built", "minutes": round((time.time() - t0) / 60, 1), "counterparties": r["rows"],
                "money out (M)": (r["out_amt"] or 0) / 1e6,
                "share matched within the window": (r["pt2_out_amt"] or 0) / (r["out_amt"] or 1),
                "switches from a TPPP": r["switch_from_tppp"]})
    handle.update(render(f"§4 Building … {m} done", ("Months", pd.DataFrame(log))))
handle.update(render(f"§4 Built {sum(x['status'] == 'built' for x in log)} months, kept "
                     f"{sum(x['status'] == 'kept' for x in log)} ({(time.time() - t_all) / 60:.1f} min)",
                     ("Months", pd.DataFrame(log)),
                     note(f"Impala, once: INVALIDATE METADATA {T_OUT};  after later runs: REFRESH {T_OUT};")))

## §5 Does the evidence separate processors?

The measures over all built months (ratio of sums, as the app computes them), for counterparties on the known-
processor seed list against all others. AUC = chance that a known processor has a higher value than another
counterparty: 0.5 means no separation, 1 perfect, below 0.5 separates the other way. The app's model learns the
direction and weight itself; this is only a check that the measures carry signal.

In [ ]:
# §5  Validation against known processor names
KNOWN_PROC_RE = (r"\b(?:ADP|PAYCHEX|PAYCOM|PAYLOCITY|DAYFORCE|CERIDIAN|GUSTO|PAYCOR|ISOLVED|TRINET|JUSTWORKS|INSPERITY|"
                 r"RIPPLING|STRIPE|PAYPAL|BRAINTREE|WORLDPAY|FISERV|FIRST DATA|GLOBAL PAYMENTS|HEARTLAND PAYMENT|TSYS|"
                 r"ELAVON|PAYMENTECH|AUTHORIZE NET|SHIFT4|NUVEI|ADYEN|PAYSAFE|REPAY|DWOLLA|BILL COM|AVIDXCHANGE|"
                 r"CLICKPAY|APPFOLIO|YARDI|PAYLEASE|ZEGO|REALPAGE)\b")
ev = (spark.table(T_OUT).where(F.col("month").isin(RUN_MONTHS)).groupBy("ent_id")
      .agg(*[F.sum(c).alias(c) for c in AMT_COLS + CNT_COLS]))
names = (spark.table(T_CPTY_MET).where(F.col("month").isin(RUN_MONTHS))
         .groupBy("ent_id").agg(F.max(F.struct("month", "ent_name")).getField("ent_name").alias("ent_name")))
E = ev.join(names, "ent_id", "left").toPandas()
nm = E.ent_name.fillna("").str.upper().str.replace(r"[^A-Z0-9]+", " ", regex=True)
E["known"] = nm.str.contains(KNOWN_PROC_RE, regex=True)
ratio = lambda a, b: E[a] / E[b].where(E[b] > 0)
M = {"money out right after money in": ratio("pt2_out_amt", "out_amt"),
     "money out on its 4 biggest days": ratio("top4_out_amt", "out_amt"),
     "money out on its main weekday": ratio("wkday_max_out_amt", "out_amt"),
     "paying organisations that also pay a registered TPPP": ratio("payer_orgs_tppp", "payer_orgs"),
     "new paying organisations that just left a registered TPPP": ratio("switch_from_tppp", "new_payer_orgs")}

def auc(x, y):
    # rank-based AUC (Mann-Whitney), NaN rows left out
    ok = x.notna()
    x, y = x[ok], y[ok]
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = x.rank()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

rows = []
for k, x in M.items():
    rows.append({"measure": k, "counterparties with a value": int(x.notna().sum()),
                 "median, known processors": x[E.known].median(), "median, others": x[~E.known].median(),
                 "AUC known vs others": auc(x, E.known)})
show("§5 Evidence vs known processor names", ("Separation", pd.DataFrame(rows)),
     note(f"{int(E.known.sum()):,} known processors among {len(E):,} counterparties over {RUN_MONTHS[0]} to "
          f"{RUN_MONTHS[-1]}."),
     note(f"Impala, once: INVALIDATE METADATA {T_OUT};  after later runs: REFRESH {T_OUT};"))